# 02 · The colexification network and typological matrices

Narrative only: the logic lives in `src/network/`. Run `python run.py network` first. Decision numbers refer to `DECISIONS.md`.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import config
from src.utils.io import read_parquet
from src.utils.concept_order import load_concept_order
from src.utils.matrices import load_matrix
load = lambda name: json.loads((config.RESULTS_DIR / name).read_text())

## 1. Our recount reproduces CLICS⁴ exactly (D-03)
With CLICS's own conventions (identity on `Segments`, every variety counted), the recount matches the shipped colexification table pair for pair.

In [ ]:
v = load('clics_validation.json'); {k: v[k] for k in v if k != 'examples_of_disagreement'}

## 2. Split source entries (D-37)
A source item such as *ARM OR HAND* is split by CLICS into one form per concept, which then counts as a colexification. The primary network requires the two concepts to have been recorded separately.

In [ ]:
ops = load('colex_operationalisations.json')
pd.DataFrame([{k: v for k, v in c.items() if k != 'largest_family_count_drops'} for c in ops['comparisons']])

In [ ]:
pd.DataFrame(ops['comparisons'][0]['largest_family_count_drops']).head(10)

## 3. The network

In [ ]:
s = load('network_summary.json')
{k: s[k] for k in ['n_pairs','n_pairs_colexified_any','n_colexified_pairs_single_family','n_edges_min_families','largest_component','isolated_concepts','degree_median','highest_degree']}

In [ ]:
pd.read_csv(config.RESULTS_DIR / 'top_edges.csv').head(20)[['gloss_a','gloss_b','n_families','n_languages','cov_families','rate_families']]

### The proposal's named pairs

In [ ]:
pairs = read_parquet(config.PROCESSED_DIR / 'pairs.parquet').set_index(['concept_a','concept_b'])
named = [('ARM','HAND'),('TREE','WOOD'),('TOMORROW','YESTERDAY'),('FINGER','TOE'),('SEE','UNDERSTAND'),('LANGUAGE','TONGUE')]
pairs.loc[named, ['n_families','n_languages','cov_families','rate_families','is_edge']]

## 4. Choosing the typological distance (D-39)
The profile-cosine metric planned in Phase 0 leaves most pairs tied at the maximum distance; weighted shortest path is graded and stable across operationalisations.

In [ ]:
from IPython.display import Image
Image(filename=str(config.FIGURES_DIR / 'figS2_network_decisions.png'), width=900)

In [ ]:
pd.read_csv(config.RESULTS_DIR / 'typological_matrix_ties.csv')

In [ ]:
c = pd.read_csv(config.RESULTS_DIR / 'typological_matrix_correlations.csv')
c[(c.matrix_a == 'shortest_path') | (c.matrix_b == 'shortest_path')]

## 5. Family subnetworks (D-09, D-25 amendment)
All five families named in the proposal qualify. Within-family matrices are > 99% ties, so H3 is tested at the pair level (AUC).

In [ ]:
fam = pd.read_csv(config.RESULTS_DIR / 'family_subnetworks.csv')
fam[fam.qualifies][['family_name','n_languages','n_concepts_attested','concept_coverage','named_in_proposal']]

In [ ]:
pd.read_csv(config.RESULTS_DIR / 'family_matrices.csv')[['family_name','n_languages','n_within_family_edges','share_pairs_tied_at_1']]

## 6. The English audit sheet
WordNet candidates that are also network edges: these are what the human audit (Phase 6) must decide.

In [ ]:
a = pd.read_csv(config.ENGLISH_AUDIT_PATH)
print(len(a), 'pairs to audit')
a.head(15)[['gloss_a','gloss_b','shared_lemma','evidence_route','n_families']]